# FlexENN: A Graph Neural Network for Binding Energy Prediction of Globular and Intrinsically Disordered Proteins

This notebook trains a graph neural network, FlexENN, to predict
the experimental binding free energy ΔG of protein–protein complexes from
their PDB structures.

**Pipeline**

1. Load metadata (PDB IDs, interacting chains, experimental ΔG) from CSV.
2. For each complex, parse the PDB, identify interface residues (Cα–Cα
   within 8 Å of the partner chain), and build a residue-level graph.
3. Encode each residue with a 29-dim feature vector (amino-acid one-hot,
   hydrophobicity, charge, φ/ψ, SASA, B-factor, interface flag, distance
   to partner chain, partner-chain flag).
4. Connect residues with intra-chain (≤ 8 Å) and inter-chain (≤ 4.5 Å) edges.
5. Train the model that passes messages along these edges and predicts a
   single scalar ΔG per graph from a mean-pooled graph embedding.

Run cells top-to-bottom. The slow step is graph construction (~minutes on
the full dataset, depending on the number of PDBs).

In [ ]:
import numpy 
#import matplotlib.pyplot
#import pandas
import torch
print(numpy.__version__)
#print(matplotlib.__version__)
#print(pandas.__version__)
print(torch.__version__)
import torchvision
print(torchvision.__version__)

import torch_geometric
print(torch_geometric.__version__)

Main imports for the rest of the notebook.

In [ ]:
import os
import math
import glob
import pandas as pd
import numpy as np
from collections import defaultdict
from Bio.PDB import PDBParser, NeighborSearch, Selection
import networkx as nx
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, global_mean_pool, MessagePassing
from torch_geometric.loader import DataLoader
from torch_scatter import scatter_mean
from torch_geometric.utils import add_self_loops
from Bio.PDB.vectors import calc_dihedral
from Bio.PDB import ShrakeRupley
from scipy.stats import pearsonr, spearmanr
from torch_geometric.utils import scatter
#import seaborn as sns
print(f"{np.__version__=}")
#print(f"{scipy.__version__=}")
from statistics import mean, stdev

## 2. Reproducibility

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # For full determinism (PyTorch)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 3. Dataset configuration

`csv_path` points to the CSV containing the columns `pdb_id`,
`interacting_chains`, and `dG_exp`. `pdb_folder` is the directory holding
the matching `<pdb_id>.pdb` files.

In [ ]:
csv_path = "../data/IDP_complexes.csv"
pdb_folder = "../data/IDP-dataset/"

df1 = pd.read_csv(csv_path)
print(df1['dG_exp'].mean())

## 4. Metadata loading

`parse_chain_groups` splits a chain string like `"AB_CD"` into two
partner-chain lists `['A','B']` and `['C','D']`.

`load_metadata` walks every row in the CSV, verifies the PDB file exists,
and returns one metadata dict per complex (PDB path, partner chains,
experimental ΔG).

In [ ]:
# loading the dataset from the CSV file
def parse_chain_groups(chain_string):
    """
    Example inputs:
    'A_B'   → ['A'], ['B']
    'AB_C'  → ['A','B'], ['C']
    'AB_CD' → ['A','B'], ['C','D']
    """
    left, right = chain_string.split("_")
    group1 = list(left)
    group2 = list(right)
    return group1, group2


def load_metadata(csv_path, pdb_folder):
    """
    Reads CSV and checks which PDBs are available.
    Returns a list of dicts, one per complex.
    """
    df = pd.read_csv(csv_path)
    dataset = []

    for _, row in df.iterrows():
        pdb_id_raw = row["pdb_id"]
        pdb_id = pdb_id_raw.lower()
        chains  = row["interacting_chains"]
        dG_exp  = float(row["dG_exp"])

        pdb_file = os.path.join(pdb_folder, f"{pdb_id}.pdb")

        if not os.path.exists(pdb_file):
            print(f"WARNING: Missing PDB file for {pdb_id}")
            continue

        g1, g2 = parse_chain_groups(chains)

        dataset.append({
            "pdb_id": pdb_id,
            "pdb_path": pdb_file,
            "partner1": g1,
            "partner2": g2,
            "dG": dG_exp
        })

    return dataset

## 5. Residue extraction from a PDB

`extract_residues` parses one PDB file and returns:

- `residues1`, `residues2`: lists of `(chain_id, resname, resseq, Cα coord)`
  tuples — one list per partner.
- `residue_objects`: a `dict[chain_id][resseq] = Bio.PDB residue`, used by
  `make_node_features` to access dihedrals, B-factors, etc.

In [ ]:
def extract_residues(pdb_path, partner1, partner2):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", pdb_path)
    model = structure[0]

    residues1 = []
    residues2 = []

    residue_objects = {}  
                          

    for chain in model:
        chain_id = chain.id
        residue_objects[chain_id] = {}

        for res in chain:
            if res.id[0] != " ":
                continue

            resseq = res.id[1]
            residue_objects[chain_id][resseq] = res 

            if "CA" not in res:
                continue

            ca = res["CA"].coord
            entry = (chain_id, res.resname, resseq, ca)

            if chain_id in partner1:
                residues1.append(entry)
            elif chain_id in partner2:
                residues2.append(entry)

    return residues1, residues2, residue_objects


## 6. Interface detection

A residue is on the interface if its Cα is within `cutoff` Å (default 8 Å)
of any Cα on the partner chain. Returns the indices into `residues1` and
`residues2`.

In [ ]:
def detect_interface(residues1, residues2, cutoff=8.0):
    """
    residues1: list from partner1
    residues2: list from partner2
    Returns:
    - interface1: indices of interface residues in partner1
    - interface2: indices of interface residues in partner2
    """

    interface1 = set()
    interface2 = set()

    # Loop through every pair of residues
    for i, r1 in enumerate(residues1):
        coord1 = r1[3]  # C-alpha coordinate
        for j, r2 in enumerate(residues2):
            coord2 = r2[3]

            dist = np.linalg.norm(coord1 - coord2)

            if dist < cutoff:
                interface1.add(i)
                interface2.add(j)

    return list(interface1), list(interface2)


## 7. Node features

Each residue is encoded as a **29-dim feature vector**:

| index   | feature                                                           |
|---------|-------------------------------------------------------------------|
| 0–19    | one-hot amino acid (ALA, ARG, …, VAL)                             |
| 20      | partner-chain flag (1 for partner 1, 0 for partner 2)             |
| 21      | Kyte–Doolittle hydrophobicity                                     |
| 22      | side-chain charge at pH 7                                         |
| 23      | φ (radians)                                                       |
| 24      | ψ (radians)                                                       |
| 25      | Bias feature      |
| 26      | Cα B-factor                                                       |
| 27      | interface flag (1 if `i in iface_set`)                            |
| 28      | minimum Cα–Cα distance to any partner residue (Å)                 |

In [ ]:
# features to add to the GNN
# Amino acid mapping for the vector representation of the residues at the interface 

AA_LIST = [
    "ALA","ARG","ASN","ASP","CYS",
    "GLN","GLU","GLY","HIS","ILE",
    "LEU","LYS","MET","PHE","PRO",
    "SER","THR","TRP","TYR","VAL"
]

AA_TO_INDEX = {aa:i for i, aa in enumerate(AA_LIST)}


# Kyte-Doolittle hydrophobicity scale
HYDRO = {
    "ALA": 1.8, "ARG": -4.5, "ASN": -3.5, "ASP": -3.5, "CYS": 2.5,
    "GLN": -3.5, "GLU": -3.5, "GLY": -0.4, "HIS": -3.2, "ILE": 4.5,
    "LEU": 3.8, "LYS": -3.9, "MET": 1.9, "PHE": 2.8, "PRO": -1.6,
    "SER": -0.8, "THR": -0.7, "TRP": -0.9, "TYR": -1.3, "VAL": 4.2
}

# Charge: +1, -1, or 0
CHARGE = {
    "ARG": +1, "LYS": +1, "HIS": +0.1,    # histidine ~10% protonated
    "ASP": -1, "GLU": -1
}

def compute_phi_psi(residue):
    try:
        # Requires N, CA, C atoms of this residue and C of prev, N of next
        n = residue['N'].get_vector()
        ca = residue['CA'].get_vector()
        c = residue['C'].get_vector()

        # previous C
        prev = residue.get_prev()
        nxt = residue.get_next()

        if prev and 'C' in prev and nxt and 'N' in nxt:
            c_prev = prev['C'].get_vector()
            n_next = nxt['N'].get_vector()

            phi = calc_dihedral(c_prev, n, ca, c)
            psi = calc_dihedral(n, ca, c, n_next)
            return float(phi), float(psi)
    except:
        pass

    return 0.0, 0.0   # default if unavailable


def make_node_features(residues, iface_set, partner_residues, partner_flag):
    """
    residues: list of tuple (chain, resname, resnum, coord)
    iface_set: a set of interface indices (you can pass None)
    partner_residues: residues from the other chain (for min distance)
    partner_flag: 1 or 0 (which partner)

    RETURNS:
        numpy array (N, 29)
    """
    feat_list = []

    for i, (chain, resname, resnum, coord) in enumerate(residues):

        vec = np.zeros(29, dtype=np.float32)

        # 1. One-hot amino acid
        if resname in AA_TO_INDEX:
            vec[AA_TO_INDEX[resname]] = 1.0

        # 2. Hydrophobicity
        vec[21] = HYDRO.get(resname, 0.0)

        # 3. Charge
        vec[22] = CHARGE.get(resname, 0.0)

        # 4/5. Phi/Psi
        phi, psi = compute_phi_psi(residue_objects[chain][resnum])
        vec[23] = phi
        vec[24] = psi

        # 6. A bias feature
        residue = residue_objects[chain][resnum]
        vec[25] = 0.0

        # 7. B-factor
        if "CA" in residue:
            vec[26] = residue["CA"].get_bfactor()
        else:
            vec[26] = 0.0

        # 8. interface flag
        if iface_set is not None:
            vec[27] = 1.0 if i in iface_set else 0.0
        else:
            vec[27] = 0.0

        # 9. min distance to partner chain CA atoms
        this_ca = coord
        dmin = min(np.linalg.norm(this_ca - p[3]) for p in partner_residues)
        vec[28] = dmin

        # 10. partner chain flag
        vec[20] = partner_flag

        feat_list.append(vec)

    return np.array(feat_list)


 

## 8. Edge construction

For each complex (after interface residues have been selected):

- **Intra-chain edges** between residues on the same partner whose Cα atoms
  are within `cutoff_intra` Å (default 8 Å).
- **Inter-chain edges** between residues on opposite partners whose Cα
  atoms are within `cutoff_inter` Å (default 4.5 Å).

Every edge is added in both directions.

In [ ]:
def build_edges(res1, res2, cutoff_intra=8.0, cutoff_inter=4.5):
    edges = []

    n1 = len(res1)
    n2 = len(res2)

    # Intra-chain edges for partner1
    for i in range(n1):
        for j in range(i+1, n1):
            if np.linalg.norm(res1[i][3] - res1[j][3]) < cutoff_intra:
                edges.append((i, j))
                edges.append((j, i))

    # Intra-chain edges for partner2
    for i in range(n2):
        for j in range(i+1, n2):
            if np.linalg.norm(res2[i][3] - res2[j][3]) < cutoff_intra:
                edges.append((n1 + i, n1 + j))
                edges.append((n1 + j, n1 + i))

    # Inter-chain edges (interface)
    for i in range(n1):
        for j in range(n2):
            if np.linalg.norm(res1[i][3] - res2[j][3]) < cutoff_inter:
                edges.append((i, n1 + j))
                edges.append((n1 + j, i))

    # optional self-loops
    # for i in range(n1 + n2):
    #     edges.append((i, i))

    return edges


## 9. Graph assembly

`build_graph_object` stacks features for partner 1 and partner 2, stores
3D coordinates (used later for relative-position edge messages), and
attaches the experimental ΔG target.

In [ ]:
def build_graph_object(res1_if, res2_if, dG, edges):
    
    # Make node features for the interface residues only
    feat1 = make_node_features(res1_if, iface_set=None,
                               partner_residues=res2_if, partner_flag=1)
    feat2 = make_node_features(res2_if, iface_set=None,
                               partner_residues=res1_if, partner_flag=0)

    x = np.vstack([feat1, feat2])

    coords = np.array([r[3] for r in res1_if + res2_if])

    edge_index = np.array(edges).T  # (2, E)

    return {
        "x": x,
        "edge_index": edge_index,
        "coords": coords,
        "y": dG
    }


## 10. Build the dataset

For every complex in the metadata, run the full pipeline (parse → interface
→ features → edges → graph) and convert the resulting dicts into
PyTorch-Geometric `Data` objects. This is the slow cell — minutes on the
full dataset.

In [ ]:
dataset = load_metadata(csv_path, pdb_folder)
print(dataset[:1])

all_graphs = []

for item in dataset:
    
    res1, res2, residue_objects = extract_residues(
        item["pdb_path"], item["partner1"], item["partner2"]
    )

    

    iface1, iface2 = detect_interface(res1, res2)

    # Get interface residues only
    res1_if = [res1[i] for i in iface1]
    res2_if = [res2[j] for j in iface2]

 
    edges = build_edges(res1_if, res2_if)


    graph = build_graph_object(
        res1_if,      
        res2_if,      
        item["dG"], 
        edges
    )

    all_graphs.append(graph)



print(f"Total graphs built: {len(all_graphs)}")
gnn_graphs = []

for g in all_graphs:
    x = torch.tensor(g["x"], dtype=torch.float32)              # (N, F)
    edge_index = torch.tensor(g["edge_index"], dtype=torch.long)  # (2, E)
    y = torch.tensor([g["y"]], dtype=torch.float32)            # (1,)
    pos = torch.tensor(g["coords"], dtype=torch.float32)       # (N, 3)  <-- coordinates

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    gnn_graphs.append(data)

print("Number of GNN graphs:", len(gnn_graphs))
print("Example shapes:",
      "x", gnn_graphs[0].x.shape,
      "pos", gnn_graphs[0].pos.shape,
      "edge_index", gnn_graphs[0].edge_index.shape)

## 11. Sanity checks

Inspect edge_index shapes and a few graphs to confirm node counts, feature
dimensions, and ΔG labels look right. The histogram cell plots the number
of edges per graph and writes them to `edge_counts_folded.csv`.

In [ ]:
for i, g in enumerate(all_graphs):
    ei = g["edge_index"]
    print(i, type(ei), np.array(ei).shape)

# Checking a few graphs for sanity
for i, g in enumerate(gnn_graphs[:3]):
    print(f"Graph {i}:")
    print("  Nodes:", g.x.shape[0])
    print("  Features:", g.x.shape[1])
    print("  Edges:", g.edge_index.shape[1])
    print("  Coordinates:", g.pos.shape)
    print("  y:", g.y.item())


## 12. GNN model

**`GNNLayer`** — one message-passing layer.

For each directed edge `(row → col)`:

1. Build an edge message from `[x[row], x[col], pos[row] - pos[col]]` and
   pass it through `edge_mlp`.
2. Aggregate incoming messages per node with `mean`.
3. Update each node by concatenating its old features with the aggregated
   message and passing through `node_mlp`.

In [ ]:
class GNNLayer(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr="mean")
        edge_in_dim = in_features * 2 + 3   # sender feat + receiver feat + relative pos
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_in_dim, out_features),
            nn.SiLU(),
            nn.Linear(out_features, out_features)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(in_features + out_features, out_features),
            nn.SiLU(),
            nn.Linear(out_features, out_features)
        )
    def forward(self, x, pos, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        row, col = edge_index
        rel_pos = pos[row] - pos[col]    # (E, 3)
        edge_feat = torch.cat([x[row], x[col], rel_pos], dim=-1)  # (E, 45)
        m = self.edge_mlp(edge_feat)
        out = self.propagate(edge_index, x=x, m=m)
        return self.node_mlp(torch.cat([x, out], dim=-1))
    def message(self, m):
        return m

**`GNNModel`** — stacks `layers` GNN layers, mean-pools all node
embeddings within a graph (`scatter_mean` on `data.batch`), and feeds the
graph embedding through a small MLP that outputs a scalar ΔG.

In [ ]:
class GNNModel(nn.Module):
    def __init__(self, in_features=29, hidden=64, layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(GNNLayer(in_features, hidden))
        for _ in range(layers - 1):
            self.layers.append(GNNLayer(hidden, hidden))
        self.readout = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )
    def forward(self, data):
        x, pos, edge_index = data.x, data.pos, data.edge_index
        for layer in self.layers:
            x = layer(x, pos, edge_index)
        graph_emb = scatter_mean(x, data.batch, dim=0)
        return self.readout(graph_emb)

## 13. DataLoader & training loop

A seeded PyG `DataLoader` shuffles graphs each epoch. Training uses Adam
with MSE loss over predicted vs. experimental ΔG.

In [ ]:
batch_size = 16
train_loader = DataLoader(
    gnn_graphs,
    batch_size=batch_size,
    shuffle=True,
    worker_init_fn=np.random.seed(42),
    generator=torch.Generator().manual_seed(42)
)


def train_gnn(model, loader, epochs=125, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for batch in loader:
            optimizer.zero_grad()
            pred = model(batch).view(-1)
            y = batch.y.view(-1)

            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()


        print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f}")

## 14. Train the model

In [ ]:
model = GNNModel(in_features=29)

train_gnn(model, train_loader)

## 15. Evaluation

Compute system-level MSE, MAE, and Pearson correlation between predicted
and experimental ΔG, then plot true vs. predicted values.

In [ ]:
def evaluate_system(model, loader):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in loader:
            pred = model(batch).view(-1)
            y = batch.y.view(-1)

            all_preds.append(pred.cpu())
            all_targets.append(y.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    # ----- SYSTEM-LEVEL METRICS -----
    mse = np.mean((all_preds - all_targets) ** 2)
    pearson_r, _ = pearsonr(all_targets, all_preds)
    mae = np.mean(np.abs(all_preds - all_targets))

    return mse, mae, pearson_r

mse, mae, pearson_r = evaluate_system(model, train_loader)

print(f"System MSE     : {mse:.6f}")
print(f"System MAE     : {mae:.6f}")
print(f"Pearson r     : {pearson_r:.4f}")

## 16. Inference on a held-out structure

Run the trained model on the benchmark set that was not seen during training. Edit
`pdb_paths` and the chain assignments inside `extract_residues` to match
your target. The cell prints one ΔG prediction per structure and (for
multiple structures) reports the mean ± SD.

In [ ]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

holdout_graphs = []

pdb_paths = ['../benchmark_set/6RF2.pdb']
for pdb_path in pdb_paths:
    pdb_id = os.path.basename(pdb_path).replace(".pdb", "")

    iface1, iface2 = detect_interface(res1, res2, cutoff=8)
    
    # Get interface-only residues
    res1_if = [res1[i] for i in iface1]
    res2_if = [res2[j] for j in iface2]

    # Build edges
    edges = build_edges(res1_if, res2_if)

    # Build graph
    g = build_graph_object(
        res1_if,
        res2_if,
        None,
        edges
    )

    g["pdb_id"] = pdb_id
    holdout_graphs.append(g)

print("Holdout graphs:", len(holdout_graphs))

In [ ]:
# Convert to PyG
holdout_pyg = []
for g in holdout_graphs:
    x = torch.tensor(g["x"], dtype=torch.float32)
    edge_index = torch.tensor(g["edge_index"], dtype=torch.long)
    pos = torch.tensor(g["coords"], dtype=torch.float32)
    y = torch.tensor([0.0])  # dummy

    data = Data(x=x, edge_index=edge_index, pos=pos, y=y)
    data.pdb_id = g["pdb_id"]
    holdout_pyg.append(data)

# Predict
predictions = {}
model.eval()
with torch.no_grad():
    for d in holdout_pyg:
        # single-graph batch
        d.batch = torch.zeros(d.x.size(0), dtype=torch.long)

        pred = model(d).item()
        predictions[d.pdb_id] = pred

print("Predictions:")
for pdb_id, pred in predictions.items():
    print(f"  {pdb_id}: {pred:.4f}")

# compute mean safely (convert dict_values to list first)
# compute mean and standard deviation safely
if len(predictions) > 0:
    pred_values = list(predictions.values())

    avg_pred = mean(pred_values)

    if len(pred_values) > 1:
        std_pred = stdev(pred_values)
    else:
        std_pred = 0.0  # stdev undefined for single value

    print(f"Mean of predictions over {len(predictions)} structures: {avg_pred:.4f}")
    print(f"Standard deviation: {std_pred:.4f}")
    print(f"Mean ± SD: {avg_pred:.4f} ± {std_pred:.4f}")
else:
    print("No predictions were made.")